In [1]:
import re
from collections import Counter
import numpy as np

In [3]:
toy_corpus = [
    "king queen prince princess royal crown throne kingdom",
    "man woman boy girl family mother father sister brother",
    "paris france berlin germany madrid spain rome italy",
    "apple orange banana fruit mango grape peach fruit",
    "dog cat animal puppy kitten pet animal",
    "car bus train vehicle road travel transport",
    "python numpy pandas code machine learning data science",
    "deep learning neural network model gradient loss optimizer",
    "king man queen woman prince boy princess girl",
    "paris city berlin city rome city madrid city",
]

raw_text = " ".join(toy_corpus).lower()

def tokenize(text):
    """
    Very simple tokenizer:
    - lowercase
    - keep alphabetic tokens
    """
    return re.findall(r"[a-z]+", text.lower())


tokens = tokenize(raw_text)

In [5]:
# Vocabulary building
def build_vocab(tokens, min_count=1):
    """
    Build:
    - word_to_id: word -> integer index
    - id_to_word: integer index -> word
    - counts: word frequencies
    """
    counts = Counter(tokens)

    # Keep words that appear at least min_count times
    vocab_words = sorted([w for w, c in counts.items() if c >= min_count])

    word_to_id = {w: i for i, w in enumerate(vocab_words)}
    id_to_word = {i: w for w, i in word_to_id.items()}

    filtered_tokens = [w for w in tokens if w in word_to_id]
    token_ids = np.array([word_to_id[w] for w in filtered_tokens], dtype=np.int64)

    return word_to_id, id_to_word, counts, token_ids


word_to_id, id_to_word, counts, token_ids = build_vocab(tokens, min_count=1)
vocab_size = len(word_to_id)

In [6]:
# Training pair generation
def generate_skipgram_pairs(token_ids, window_size=2):
    """
    For each center word, create (center, context) pairs
    using a symmetric context window.
    """
    pairs = []

    for i, center_id in enumerate(token_ids):
        left = max(0, i - window_size)
        right = min(len(token_ids), i + window_size + 1)

        for j in range(left, right):
            if i == j:
                continue
            context_id = token_ids[j]
            pairs.append((center_id, context_id))

    return np.array(pairs, dtype=np.int64)


pairs = generate_skipgram_pairs(token_ids, window_size=2)

In [7]:
# Negative sampling distribution

def build_negative_sampling_distribution(counts, word_to_id, power=0.75):
    """
    Word2vec commonly samples negative words from a smoothed unigram
    distribution proportional to count(word)^0.75.
    """
    freqs = np.zeros(len(word_to_id), dtype=np.float64)

    for word, idx in word_to_id.items():
        freqs[idx] = counts[word]

    probs = freqs ** power
    probs /= probs.sum()
    return probs


neg_sampling_probs = build_negative_sampling_distribution(counts, word_to_id, power=0.75)

In [8]:
# Math helpers

def sigmoid(x):
    """
    Numerically stable sigmoid.
    """
    x = np.clip(x, -10, 10)
    return 1.0 / (1.0 + np.exp(-x))


def sample_negative_indices(rng, probs, num_negatives, forbidden_idx):
    """
    Draw negative samples.
    We avoid the true context word for clarity.
    Duplicates are allowed (as in many practical implementations).
    """
    negatives = []
    vocab_size = len(probs)

    while len(negatives) < num_negatives:
        idx = rng.choice(vocab_size, p=probs)
        if idx != forbidden_idx:
            negatives.append(idx)

    return np.array(negatives, dtype=np.int64)

In [9]:
# Core training loop
def train_skipgram_negative_sampling(
    pairs,
    vocab_size,
    embedding_dim=20,
    learning_rate=0.05,
    epochs=200,
    num_negatives=5,
    neg_sampling_probs=None,
    seed=42,
    verbose=True,
):
    """
    Pure NumPy training for Skip-Gram with Negative Sampling (SGNS).

    Parameters learned:
    - W_in  : center-word embeddings, shape (V, D)
    - W_out : context-word embeddings, shape (V, D)

    For one positive pair (center=c, context=o), the loss is:

        L = -log(sigmoid(u_o^T v_c)) - sum_i log(sigmoid(-u_n_i^T v_c))

    where:
    - v_c is the center embedding
    - u_o is the positive context embedding
    - u_n_i are negative-context embeddings
    """
    rng = np.random.default_rng(seed)

    # Small random initialization
    W_in = rng.normal(0.0, 0.1, size=(vocab_size, embedding_dim))
    W_out = rng.normal(0.0, 0.1, size=(vocab_size, embedding_dim))

    if neg_sampling_probs is None:
        neg_sampling_probs = np.ones(vocab_size) / vocab_size

    for epoch in range(1, epochs + 1):
        # Shuffle training pairs each epoch
        shuffled_indices = rng.permutation(len(pairs))
        epoch_loss = 0.0

        for idx in shuffled_indices:
            center_id, context_id = pairs[idx]

            # Draw K negative samples
            negative_ids = sample_negative_indices(
                rng=rng,
                probs=neg_sampling_probs,
                num_negatives=num_negatives,
                forbidden_idx=context_id,
            )

            # Save current vectors before updating
            v_c = W_in[center_id].copy()             # center vector
            u_o = W_out[context_id].copy()           # positive context vector
            u_neg = W_out[negative_ids].copy()       # negative context vectors, shape (K, D)

            # Forward pass
            # Positive score: should become large
            pos_score = np.dot(u_o, v_c)             # scalar

            # Negative scores: should become small
            neg_scores = u_neg @ v_c                # shape (K,)

            # Sigmoid probabilities
            pos_sigmoid = sigmoid(pos_score)        # sigma(u_o^T v_c)
            neg_sigmoid = sigmoid(neg_scores)       # sigma(u_n^T v_c)

            # Loss
            # Add a tiny epsilon inside log for numerical stability
            eps = 1e-12
            loss = -np.log(pos_sigmoid + eps) - np.sum(np.log(1.0 - neg_sigmoid + eps))
            epoch_loss += loss

            # Gradients
            # For positive term:
            # d/ds [-log(sigmoid(s))] = sigmoid(s) - 1
            g_pos = pos_sigmoid - 1.0               # scalar

            # For negative term:
            # d/ds [-log(sigmoid(-s))] = sigmoid(s)
            g_neg = neg_sigmoid                  

            # Gradient wrt center vector v_c
            grad_v_c = g_pos * u_o + np.sum(g_neg[:, None] * u_neg, axis=0)

            # Gradient wrt positive context vector u_o
            grad_u_o = g_pos * v_c

            # Gradient wrt each negative context vector
            grad_u_neg = g_neg[:, None] * v_c[None, :]

            # SGD parameter update
            W_in[center_id] -= learning_rate * grad_v_c
            W_out[context_id] -= learning_rate * grad_u_o

            np.add.at(W_out, negative_ids, -learning_rate * grad_u_neg)

        if verbose and (epoch == 1 or epoch % 20 == 0):
            avg_loss = epoch_loss / len(pairs)
            print(f"Epoch {epoch:3d} | average loss = {avg_loss:.4f}")

    return W_in, W_out


In [10]:
# Training
embedding_dim = 20
learning_rate = 0.05
epochs = 200
num_negatives = 5

W_in, W_out = train_skipgram_negative_sampling(
    pairs=pairs,
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    learning_rate=learning_rate,
    epochs=epochs,
    num_negatives=num_negatives,
    neg_sampling_probs=neg_sampling_probs,
    seed=42,
    verbose=True,
)

Epoch   1 | average loss = 4.1549
Epoch  20 | average loss = 1.6041
Epoch  40 | average loss = 1.0226
Epoch  60 | average loss = 0.9321
Epoch  80 | average loss = 0.9061
Epoch 100 | average loss = 0.9653
Epoch 120 | average loss = 1.0027
Epoch 140 | average loss = 0.9977
Epoch 160 | average loss = 0.9183
Epoch 180 | average loss = 0.9293
Epoch 200 | average loss = 0.9201


In [11]:
# Final word vectors
def get_final_embeddings(W_in, W_out, method="input"):
    """
    Common choices:
    - 'input'  : use only center embeddings
    - 'output' : use only context embeddings
    - 'mean'   : average both matrices
    """
    if method == "input":
        return W_in
    elif method == "output":
        return W_out
    elif method == "mean":
        return 0.5 * (W_in + W_out)
    else:
        raise ValueError("method must be 'input', 'output', or 'mean'")


embeddings = get_final_embeddings(W_in, W_out, method="input")

In [12]:
# Similarity helpers
def cosine_similarity(a, b, eps=1e-12):
    return np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + eps)


def most_similar(query_word, embeddings, word_to_id, id_to_word, top_k=5):
    """
    Return the nearest neighbors of query_word by cosine similarity.
    """
    if query_word not in word_to_id:
        raise ValueError(f"'{query_word}' not in vocabulary.")

    query_id = word_to_id[query_word]
    query_vec = embeddings[query_id]

    sims = []
    for idx in range(len(id_to_word)):
        if idx == query_id:
            continue
        sim = cosine_similarity(query_vec, embeddings[idx])
        sims.append((id_to_word[idx], sim))

    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:top_k]

In [13]:
# Demo

test_words = ["king", "queen", "paris", "fruit", "python", "animal"]

for w in test_words:
    print(f"\nNearest words to '{w}':")
    for neighbor, score in most_similar(w, embeddings, word_to_id, id_to_word, top_k=5):
        print(f"  {neighbor:10s}  {score:.4f}")



Nearest words to 'king':
  woman       0.7208
  gradient    0.6670
  optimizer   0.6053
  princess    0.5754
  queen       0.5516

Nearest words to 'queen':
  boy         0.7842
  optimizer   0.6758
  royal       0.6364
  kingdom     0.5962
  woman       0.5865

Nearest words to 'paris':
  germany     0.6186
  city        0.5815
  boy         0.5756
  father      0.5547
  brother     0.5516

Nearest words to 'fruit':
  mango       0.7305
  peach       0.7241
  apple       0.6307
  banana      0.6278
  grape       0.6254

Nearest words to 'python':
  road        0.7440
  numpy       0.7136
  code        0.7105
  transport   0.7039
  pandas      0.5767

Nearest words to 'animal':
  puppy       0.7036
  pet         0.6829
  train       0.6451
  kitten      0.5907
  cat         0.5646
